In [1]:
from datetime import date, datetime
from dateutil.relativedelta import relativedelta
from functools import partial
from tenacity import retry, stop_after_delay, wait_fixed, retry_if_exception_type
from eutils import EutilsNCBIError, EutilsRequestError
import os
import pandas as pd 
import numpy as np 

from metapub import PubMedFetcher 

In [2]:
# Define NCBI error handling decorator with tenacity
retry_on_communication_error = partial(
    retry,
    stop=stop_after_delay(10),  # max. 10 seconds wait.
    wait=wait_fixed(0.4),  # wait 400ms 
    retry=retry_if_exception_type([EutilsNCBIError, EutilsRequestError])
)()

#Create function to retrieve ALL pmids (with NCBI error handling)
@retry_on_communication_error
def Get_list(query):
    fetch = PubMedFetcher()  
    num_of_articles = 500
    start_index = 0
    pmids = []
    while True:
        pmid_batch = fetch.pmids_for_query(query, 
                                      retstart=start_index,
                                      retmax=num_of_articles,
                                      pmc_only= True)
        pmids.extend(pmid_batch)
        start_index = len(pmids)
        if len(pmid_batch) < num_of_articles:
            break     
    return(pmids)

In [7]:
#Read-in query version
with open("PUBMED_query_v1.2", "r") as f:
    file = []
    for line in f:
        file.append(line.replace('\t','').replace('\n','').strip())
query = " ".join(file[1:])

In [8]:
#Run query

a = datetime.now()
START = "2000-01-01"
STOP = "2024-04-01"
start_date_str = START
pmid_list = []
while True: #define periods so that <10,000 are retrieved in the least request calls (best speed)
    if date.fromisoformat(start_date_str) <= date.fromisoformat("2002-07-01"): 
        month_interval = 6
    elif date.fromisoformat(start_date_str) <= date.fromisoformat("2005-11-01"):
        month_interval = 5
    elif (date.fromisoformat(start_date_str) <= date.fromisoformat("2009-11-01")):
        month_interval = 4
    elif (date.fromisoformat(start_date_str) <= date.fromisoformat("2011-10-01")):
        month_interval = 3
    elif (date.fromisoformat(start_date_str) <= date.fromisoformat("2023-01-01")):
        month_interval = 2
    else:
        month_interval = 4
    next_start = date.fromisoformat(start_date_str) + relativedelta(months=month_interval)
    end_date = (next_start - relativedelta(days=1))
    end_date_str = end_date.strftime('%Y-%m-%d')
    date_str = f'''(("{start_date_str}"[Date - Publication] : "{end_date_str}"[Date - Publication]) '''
    pmids = Get_list(date_str+query)
    pmids_s = list(set(pmids))
    print(start_date_str,"-", end_date_str,": ", len(pmids_s), f"({len(pmids)})") #checks the number of PMIDs for each quarter(<10,000)
    pmid_list.extend(pmids_s)
    start_date_str = next_start.strftime('%Y-%m-%d')
    if next_start>=date.fromisoformat(STOP):
        end_date_str = STOP
        break
print("Total query duration: ", datetime.now()-a)

2000-01-01 - 2000-06-30 :  1389 (1389)
2000-07-01 - 2000-12-31 :  1514 (1514)
2001-01-01 - 2001-06-30 :  1525 (1525)
2001-07-01 - 2001-12-31 :  1611 (1611)
2002-01-01 - 2002-06-30 :  1648 (1648)
2002-07-01 - 2002-12-31 :  1696 (1696)
2003-01-01 - 2003-05-31 :  1475 (1475)
2003-06-01 - 2003-10-31 :  1563 (1563)
2003-11-01 - 2004-03-31 :  1634 (1634)
2004-04-01 - 2004-08-31 :  1616 (1616)
2004-09-01 - 2005-01-31 :  1912 (1912)
2005-02-01 - 2005-06-30 :  1894 (1894)
2005-07-01 - 2005-11-30 :  2038 (2038)
2005-12-01 - 2006-03-31 :  2024 (2024)
2006-04-01 - 2006-07-31 :  1786 (1786)
2006-08-01 - 2006-11-30 :  2132 (2132)
2006-12-01 - 2007-03-31 :  2848 (2848)
2007-04-01 - 2007-07-31 :  2543 (2543)
2007-08-01 - 2007-11-30 :  2693 (2693)
2007-12-01 - 2008-03-31 :  3229 (3229)
2008-04-01 - 2008-07-31 :  3780 (3780)
2008-08-01 - 2008-11-30 :  4044 (4044)
2008-12-01 - 2009-03-31 :  4428 (4428)
2009-04-01 - 2009-07-31 :  4229 (4229)
2009-08-01 - 2009-11-30 :  4302 (4302)
2009-12-01 - 2010-02-28 :

In [11]:
print(len(pmid_clean))

234661


In [1]:
pmid_clean = set(pmid_list)
pmid_clean_list = list(pmid_clean)

# Create directories if they don't exist
os.makedirs("PMIDPMConly_lists", exist_ok=True)

# Create a date tag for the filename
date_tag = datetime.now().isoformat()[:10]

# Save PMIDs in a text file
text_file_path = os.path.join("PMIDPMConly_lists", "pmidsPMC_" + date_tag + ".txt")
np.savetxt(text_file_path, pmid_clean_list, fmt='%s', delimiter=",")

# Save PMIDs in an .npy file
npy_file_path = os.path.join("PMIDPMConly_lists", "pmidsPMC_" + ".npy")
np.save(npy_file_path, pmid_clean_list)

NameError: name 'pmid_list' is not defined